# Tournament results analysis

Notebook for inspecting outputs produced by the tournament pipeline. It expects CSV/JSONL files created by:

1. `scripts/run_tournament.py`
2. `scripts/analyze_tournament.py`
3. optionally `scripts/score_blunders.py`

Change `RESULTS_DIR` to analyze another run.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
RESULTS_DIR = PROJECT_ROOT / "results" / "e2e_smoke_runner"
RESULTS_DIR

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
def read_csv_if_exists(name: str) -> pd.DataFrame:
    path = RESULTS_DIR / name
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)

games = read_csv_if_exists("games.csv")
standings = read_csv_if_exists("standings.csv")
pair_summary = read_csv_if_exists("pair_summary.csv")
agent_summary = read_csv_if_exists("agent_summary.csv")
matchup_summary = read_csv_if_exists("matchup_summary.csv")
llm_summary = read_csv_if_exists("llm_summary.csv")
blunders = read_csv_if_exists("blunders.csv")
blunder_summary = read_csv_if_exists("blunder_summary.csv")

print("games", len(games))
print("moves scored for blunders", len(blunders))

## Standings

In [ ]:
if not agent_summary.empty:
    cols = [
        "rank", "agent", "kind", "games", "wins", "losses", "draws",
        "win_rate", "draw_rate", "avg_moves_per_game", "avg_decision_seconds",
        "tree_growth", "avg_legal_moves",
    ]
    existing = [col for col in cols if col in agent_summary.columns]
    display(
        agent_summary[existing]
        .sort_values("rank")
        .style.format({
            "win_rate": "{:.1%}",
            "draw_rate": "{:.1%}",
            "avg_moves_per_game": "{:.2f}",
            "avg_decision_seconds": "{:.4f}",
            "avg_legal_moves": "{:.2f}",
        })
    )

In [ ]:
if not agent_summary.empty:
    plot_df = agent_summary.sort_values("win_rate", ascending=True)
    ax = plot_df.plot.barh(x="agent", y="win_rate", legend=False, figsize=(8, max(4, len(plot_df) * 0.45)))
    ax.set_xlim(0, 1)
    ax.set_xlabel("Win rate")
    ax.set_ylabel("")
    ax.set_title("Overall win rate")
    plt.tight_layout()
    plt.show()

## Matchups and Side Balance

In [ ]:
if not matchup_summary.empty:
    cols = [
        "pair_id", "left", "right", "games", "left_wins", "right_wins", "draws",
        "left_win_rate", "right_win_rate", "draw_rate", "avg_moves",
        "left_avg_decision_seconds", "right_avg_decision_seconds",
    ]
    existing = [col for col in cols if col in matchup_summary.columns]
    display(
        matchup_summary[existing]
        .sort_values("pair_id")
        .style.format({
            "left_win_rate": "{:.1%}",
            "right_win_rate": "{:.1%}",
            "draw_rate": "{:.1%}",
            "avg_moves": "{:.2f}",
            "left_avg_decision_seconds": "{:.4f}",
            "right_avg_decision_seconds": "{:.4f}",
        })
    )

In [ ]:
if not agent_summary.empty and {"win_rate_as_red", "win_rate_as_yellow"}.issubset(agent_summary.columns):
    side_df = agent_summary.melt(
        id_vars=["agent"],
        value_vars=["win_rate_as_red", "win_rate_as_yellow"],
        var_name="side",
        value_name="side_win_rate",
    )
    side_df["side"] = side_df["side"].map({"win_rate_as_red": "red", "win_rate_as_yellow": "yellow"})
    pivot = side_df.pivot(index="agent", columns="side", values="side_win_rate")
    ax = pivot.plot.bar(figsize=(9, 4))
    ax.set_ylim(0, 1)
    ax.set_ylabel("Win rate")
    ax.set_title("Side balance")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## Decision Time

In [ ]:
if not agent_summary.empty:
    time_df = agent_summary.copy()
    time_df["avg_decision_ms"] = time_df["avg_decision_seconds"] * 1000
    ax = time_df.sort_values("avg_decision_ms").plot.barh(
        x="agent",
        y="avg_decision_ms",
        legend=False,
        figsize=(8, max(4, len(time_df) * 0.45)),
    )
    ax.set_xlabel("Average decision time [ms]")
    ax.set_ylabel("")
    ax.set_title("Decision cost")
    plt.tight_layout()
    plt.show()

## Blunder Rate

In [ ]:
if not blunder_summary.empty:
    cols = [
        "agent", "scored_moves", "moves_with_known_value", "blunders", "blunder_rate",
        "avg_regret", "avg_chosen_value", "avg_best_value",
    ]
    existing = [col for col in cols if col in blunder_summary.columns]
    display(
        blunder_summary[existing]
        .sort_values("blunder_rate")
        .style.format({
            "blunder_rate": "{:.1%}",
            "avg_regret": "{:.3f}",
            "avg_chosen_value": "{:.3f}",
            "avg_best_value": "{:.3f}",
        })
    )
else:
    print("No blunder_summary.csv found. Run scripts/score_blunders.py first.")

In [ ]:
if not blunder_summary.empty:
    plot_df = blunder_summary.sort_values("blunder_rate", ascending=False)
    ax = plot_df.plot.barh(x="agent", y="blunder_rate", legend=False, figsize=(8, max(4, len(plot_df) * 0.45)))
    ax.set_xlim(0, 1)
    ax.set_xlabel("Blunder rate")
    ax.set_ylabel("")
    ax.set_title("Oracle-scored blunders")
    plt.tight_layout()
    plt.show()

## LLM Reliability

In [ ]:
if not llm_summary.empty:
    display(
        llm_summary.sort_values("llm_invalid_response_rate", ascending=False)
        .style.format({
            "llm_invalid_response_rate": "{:.1%}",
            "llm_fallback_rate": "{:.1%}",
            "avg_decision_seconds": "{:.4f}",
            "win_rate": "{:.1%}",
        })
    )
else:
    print("No LLM rows in this run.")

## Raw Game Samples

In [ ]:
if not games.empty:
    cols = [
        "game_id", "pair_id", "seed", "red_agent", "yellow_agent", "winner_agent",
        "red_lines", "yellow_lines", "moves", "red_avg_decision_seconds", "yellow_avg_decision_seconds",
    ]
    existing = [col for col in cols if col in games.columns]
    display(games[existing].head(20))